# Phase 2: Gold Standard Fine-Tuning

Fine-tune the Phase 1 silver-trained Longformer on ~2,740 human-annotated gold standard articles.

**Starting Point:** `trained_models/checkpoint_epoch_4/checkpoint_epoch_4/`

**Validation Strategy:** 5-Fold Cross-Validation

**Experiments:**
1. Focal Loss gamma sweep (gamma = 1, 2, 3, 5)
2. Standard BCE comparison
3. Threshold optimization on out-of-fold predictions
4. Final model training on 100% data

In [81]:
# Imports
import os
import json
import copy
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from transformers import (
    LongformerForSequenceClassification,
    LongformerTokenizer,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from tqdm.auto import tqdm
import wandb
import psycopg2
from dotenv import load_dotenv

load_dotenv()

# Verify GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4070 Ti SUPER
VRAM: 17.2 GB


In [82]:
# ============================================================
# CONFIGURATION - Change these for different experiment variants
# ============================================================

CONFIG = {
    # Model
    "model_checkpoint": "../trained_models/checkpoint_epoch_4/checkpoint_epoch_4/",
    "num_labels": 15,
    "global_attention": "cls_plus_topic",
    
    # Training
    "max_length": 2048,
    "batch_size": 2,
    "grad_accum_steps": 8,  # effective batch size = 16
    "learning_rate": 2e-5, # changed from 2e-5
    "weight_decay": 0.01,
    "epochs": 3,
    "warmup_ratio": 0.1,
    
    # Loss - will be updated per experiment
    "loss_type": "focal",  # 'focal' or 'bce'
    "focal_gamma": 2.0,    # will sweep [1, 2, 3]
    
    # CV
    "n_folds": 5,
    "random_seed": 42,
}

# Run naming
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M")
RUN_NAME = f"gold-phase2-{TIMESTAMP}"
SAVE_DIR = Path(f"saved_models/phase2_gold_{TIMESTAMP}")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Labels
LABELS = [
    'Economic',
    'Capacity and resources',
    'Morality',
    'Fairness and equality',
    'Legality, constitutionality and jurisprudence',
    'Policy prescription and evaluation',
    'Crime and punishment',
    'Security and defense',
    'Health and safety',
    'Quality of life',
    'Cultural identity',
    'Public opinion',
    'Political',
    'External regulation and reputation',
    'Other',
]

print(f"Run name: {RUN_NAME}")
print(f"Save directory: {SAVE_DIR}")
print(f"Effective batch size: {CONFIG['batch_size'] * CONFIG['grad_accum_steps']}")

Run name: gold-phase2-20260206_0334
Save directory: saved_models\phase2_gold_20260206_0334
Effective batch size: 16


In [83]:
# ============================================================
# Initialize W&B Run
# ============================================================

run = wandb.init(
    entity="ryrousseau-london-school-of-economics-and-political-science",
    project="frame-delta-gold",
    name=RUN_NAME,
    config=CONFIG
)

print(f"W&B run initialized: {run.name}")
print(f"View at: {run.url}")

avg_labels_per_article,▁
dataset_size,▁
test_size,▁
train_size,▁
avg_labels_per_article,4.14124
dataset_size,2740
test_size,274
train_size,2466


W&B run initialized: gold-phase2-20260206_0334
View at: https://wandb.ai/ryrousseau-london-school-of-economics-and-political-science/frame-delta-gold/runs/aagj295x


## 1. Load Gold Data from Database

In [84]:
# Load gold data from database
conn = psycopg2.connect(
    dbname=os.getenv('DB_NAME'),
    user=os.getenv('DB_USER'),
    password=os.getenv('DB_PASSWORD'),
    host=os.getenv('DB_HOST'),
    port=os.getenv('DB_PORT'),
)
cur = conn.cursor()
cur.execute("""
    SELECT id, source, article_id, formatted_text, labels_idx_json
    FROM gold_train_data
""")
rows = cur.fetchall()
cur.close()
conn.close()

gold_df = pd.DataFrame(rows, columns=['id', 'source', 'article_id', 'text', 'labels_idx_json'])
gold_df['labels_idx'] = gold_df['labels_idx_json'].apply(json.loads)

print(f"Loaded {len(gold_df)} gold articles")
print(f"\nSource distribution:")
print(gold_df['source'].value_counts())

# Verify text format
sample = gold_df.iloc[0]['text']
print(f"\nSample text starts with: {sample[:50]}...")
assert sample.startswith("TOPIC:"), "Text format mismatch - must start with TOPIC:"
assert not sample.startswith("TOPIC: "), "Text format mismatch - should NOT have space after colon"
print("Format verification PASSED")

# Log dataset info to W&B
run.log({"dataset_size": len(gold_df)})

Loaded 2740 gold articles

Source distribution:
source
mfc        2224
semeval     516
Name: count, dtype: int64

Sample text starts with: TOPIC:Immigration
THE FINE PRINT: A close look at ...
Format verification PASSED


In [85]:
# Analyze label distribution
all_labels = []
for labels in gold_df['labels_idx']:
    all_labels.extend(labels)

label_counts = Counter(all_labels)
print("Label distribution:")
print("-" * 60)
for idx in range(CONFIG['num_labels']):
    count = label_counts.get(idx, 0)
    pct = count / len(gold_df) * 100
    print(f"{idx:2d}. {LABELS[idx]:45s}: {count:4d} ({pct:5.1f}%)")

avg_labels = len(all_labels) / len(gold_df)
print(f"\nAvg labels per article: {avg_labels:.2f}")

# Log to W&B
run.log({"avg_labels_per_article": avg_labels})

Label distribution:
------------------------------------------------------------
 0. Economic                                     :  870 ( 31.8%)
 1. Capacity and resources                       :  276 ( 10.1%)
 2. Morality                                     :  590 ( 21.5%)
 3. Fairness and equality                        :  599 ( 21.9%)
 4. Legality, constitutionality and jurisprudence: 1600 ( 58.4%)
 5. Policy prescription and evaluation           : 1127 ( 41.1%)
 6. Crime and punishment                         :  879 ( 32.1%)
 7. Security and defense                         :  451 ( 16.5%)
 8. Health and safety                            :  664 ( 24.2%)
 9. Quality of life                              :  825 ( 30.1%)
10. Cultural identity                            :  737 ( 26.9%)
11. Public opinion                               :  688 ( 25.1%)
12. Political                                    : 1373 ( 50.1%)
13. External regulation and reputation           :  364 ( 13.3%)
14. Other

In [86]:
# ============================================================
# TRAIN/TEST SPLIT - Hold out 10% for final unbiased evaluation
# ============================================================
from sklearn.model_selection import train_test_split

# Get primary label for stratification
def get_primary_label(labels_idx):
    """Get first label for stratification."""
    if not labels_idx:
        return 0
    return labels_idx[0]

gold_df['primary_label'] = gold_df['labels_idx'].apply(get_primary_label)

# Stratified split: 90% train, 10% test
train_df, test_df = train_test_split(
    gold_df, 
    test_size=0.10, 
    random_state=CONFIG['random_seed']
)

# Reset indices for clean indexing
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train set: {len(train_df)} articles (90%)")
print(f"Test set:  {len(test_df)} articles (10%) - HELD OUT for final evaluation")
print(f"\nTrain source distribution:")
print(train_df['source'].value_counts())
print(f"\nTest source distribution:")
print(test_df['source'].value_counts())

# Log to W&B
run.log({
    "train_size": len(train_df),
    "test_size": len(test_df),
})

Train set: 2466 articles (90%)
Test set:  274 articles (10%) - HELD OUT for final evaluation

Train source distribution:
source
mfc        1993
semeval     473
Name: count, dtype: int64

Test source distribution:
source
mfc        231
semeval     43
Name: count, dtype: int64


## 2. Dataset and DataLoader

In [87]:
class GoldFramingDataset(Dataset):
    """Dataset for gold standard framing data."""
    
    def __init__(self, texts, labels_list, tokenizer, max_length, num_labels=15):
        self.texts = texts
        self.labels_list = labels_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.num_labels = num_labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label_indices = self.labels_list[idx]
        
        # Tokenize
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        # Create multi-hot label vector
        labels = torch.zeros(self.num_labels)
        for label_idx in label_indices:
            if label_idx < self.num_labels:
                labels[label_idx] = 1.0
        
        # Set global attention on CLS and topic token (position 3)
        # This matches the cls_plus_topic mode from silver training
        global_attention_mask = torch.zeros(self.max_length, dtype=torch.long)
        global_attention_mask[0] = 1  # CLS token
        global_attention_mask[3] = 1  # First topic token after "TOPIC:"
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'global_attention_mask': global_attention_mask,
            'labels': labels
        }

print("Dataset class defined")

Dataset class defined


## 3. Focal Loss Implementation

In [88]:
class FocalLoss(nn.Module):
    """Focal Loss for multi-label classification.
    
    Focal Loss down-weights well-classified examples and focuses on hard examples.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    
    Args:
        alpha: Per-class weights (optional)
        gamma: Focusing parameter. Higher gamma = more focus on hard examples.
               gamma=0 is equivalent to BCE
               gamma=2 is common default
    """
    
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, logits, targets):
        # Binary cross entropy (unreduced)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        
        # p_t = probability of correct class
        pt = torch.exp(-bce)
        
        # Focal weight: (1 - p_t)^gamma
        focal_weight = (1 - pt) ** self.gamma
        
        # Apply focal weight
        focal_loss = focal_weight * bce
        
        # Apply per-class weights if provided
        if self.alpha is not None:
            focal_loss = self.alpha * focal_loss
        
        # Reduce
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss

print("Focal Loss class defined")

Focal Loss class defined


## 4. Create 5-Fold CV Splits

In [89]:
# Create stratified folds ON TRAIN SET ONLY (test set is held out)
# Uses primary label for stratification (approximation for multi-label)

# train_df already has primary_label from the split cell

# Create folds on train_df
skf = StratifiedKFold(n_splits=CONFIG['n_folds'], shuffle=True, random_state=CONFIG['random_seed'])

folds = []
for fold_idx, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['primary_label'])):
    folds.append({
        'fold': fold_idx,
        'train_idx': train_idx.tolist(),
        'val_idx': val_idx.tolist()
    })
    print(f"Fold {fold_idx}: train={len(train_idx)}, val={len(val_idx)}")

print(f"\nTotal train set samples: {len(train_df)} (held-out test: {len(test_df)})")

# Save fold indices for reproducibility
fold_path = Path('../data/gold_cv_folds.json')
fold_path.parent.mkdir(exist_ok=True)
with open(fold_path, 'w') as f:
    json.dump(folds, f, indent=2)
print(f"Fold indices saved to {fold_path}")

Fold 0: train=1972, val=494
Fold 1: train=1973, val=493
Fold 2: train=1973, val=493
Fold 3: train=1973, val=493
Fold 4: train=1973, val=493

Total train set samples: 2466 (held-out test: 274)
Fold indices saved to ..\data\gold_cv_folds.json


c:\Users\rhrou\miniconda3\envs\torch-gpu\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


## 5. Training Functions

In [90]:
def train_epoch(model, dataloader, optimizer, scheduler, loss_fn, device, accumulation_steps):
    """Train for one epoch."""
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    pbar = tqdm(dataloader, desc="Training")
    for step, batch in enumerate(pbar):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        global_attention_mask = batch['global_attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        
        loss = loss_fn(outputs.logits, labels)
        loss = loss / accumulation_steps
        loss.backward()
        
        total_loss += loss.item() * accumulation_steps
        
        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        
        pbar.set_postfix({'loss': f'{total_loss / (step + 1):.4f}'})
    
    return total_loss / len(dataloader)


def evaluate(model, dataloader, loss_fn, device, threshold=0.5):
    """Evaluate model and return metrics."""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    all_logits = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            global_attention_mask = batch['global_attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                global_attention_mask=global_attention_mask
            )
            
            loss = loss_fn(outputs.logits, labels)
            total_loss += loss.item()
            
            probs = torch.sigmoid(outputs.logits)
            preds = (probs > threshold).float()
            
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_logits.append(outputs.logits.cpu().numpy())
    
    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    all_logits = np.vstack(all_logits)
    
    # Calculate metrics
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return {
        'loss': total_loss / len(dataloader),
        'micro_f1': micro_f1,
        'macro_f1': macro_f1,
        'predictions': all_preds,
        'labels': all_labels,
        'logits': all_logits
    }

print("Training functions defined")

Training functions defined


In [91]:
def train_fold(fold_idx, train_idx, val_idx, df, config, loss_type='focal', gamma=2.0):
    """Train a single fold and return metrics.
    
    Args:
        fold_idx: Fold number
        train_idx: Indices for training within df
        val_idx: Indices for validation within df
        df: DataFrame to index into (train_df, NOT gold_df)
        config: Training configuration
        loss_type: 'focal' or 'bce'
        gamma: Focal loss gamma parameter
    """
    print(f"\n{'='*60}")
    print(f"FOLD {fold_idx} | Loss: {loss_type} | Gamma: {gamma if loss_type == 'focal' else 'N/A'}")
    print(f"{'='*60}")
    
    # Load fresh model for each fold
    model = LongformerForSequenceClassification.from_pretrained(
        config['model_checkpoint'],
        num_labels=config['num_labels'],
        problem_type="multi_label_classification"
    )
    model.to(device)
    
    tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096') # changed from config['model_checkpoint'] 
    
    # Create datasets
    train_texts = df.iloc[train_idx]['text'].tolist()
    train_labels = df.iloc[train_idx]['labels_idx'].tolist()
    val_texts = df.iloc[val_idx]['text'].tolist()
    val_labels = df.iloc[val_idx]['labels_idx'].tolist()
    
    train_dataset = GoldFramingDataset(train_texts, train_labels, tokenizer, config['max_length'], config['num_labels'])
    val_dataset = GoldFramingDataset(val_texts, val_labels, tokenizer, config['max_length'], config['num_labels'])
    
    train_loader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config['batch_size'])
    
    # Loss function
    if loss_type == 'focal':
        loss_fn = FocalLoss(gamma=gamma)
    else:
        loss_fn = nn.BCEWithLogitsLoss()
    
    # Optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=config['learning_rate'], weight_decay=config['weight_decay'])
    
    total_steps = len(train_loader) * config['epochs'] // config['grad_accum_steps']
    warmup_steps = int(total_steps * config['warmup_ratio'])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    
    # Training loop
    best_micro_f1 = 0
    best_metrics = None
    
    for epoch in range(config['epochs']):
        print(f"\nEpoch {epoch + 1}/{config['epochs']}")
        
        train_loss = train_epoch(
            model, train_loader, optimizer, scheduler, loss_fn, 
            device, config['grad_accum_steps']
        )
        
        val_metrics = evaluate(model, val_loader, loss_fn, device)
        
        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val Loss: {val_metrics['loss']:.4f}")
        print(f"Val Micro F1: {val_metrics['micro_f1']:.4f}")
        print(f"Val Macro F1: {val_metrics['macro_f1']:.4f}")
        
        # Log to W&B
        log_prefix = f"{loss_type}_gamma{gamma}" if loss_type == 'focal' else 'bce'
        run.log({
            f"{log_prefix}/fold{fold_idx}/epoch": epoch + 1,
            f"{log_prefix}/fold{fold_idx}/train_loss": train_loss,
            f"{log_prefix}/fold{fold_idx}/val_loss": val_metrics['loss'],
            f"{log_prefix}/fold{fold_idx}/val_micro_f1": val_metrics['micro_f1'],
            f"{log_prefix}/fold{fold_idx}/val_macro_f1": val_metrics['macro_f1'],
        })
        
        if val_metrics['micro_f1'] > best_micro_f1:
            best_micro_f1 = val_metrics['micro_f1']
            best_metrics = val_metrics.copy()
    
    # Clean up
    del model
    torch.cuda.empty_cache()
    
    return best_metrics

print("Fold training function defined")

Fold training function defined


## 6. Focal Loss Gamma Sweep

In [92]:
# Clear GPU memory
import gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()



In [93]:
# Grid search over gamma values - ON TRAIN SET ONLY
# GAMMA_VALUES = [1, 2, 3]

# all_experiment_results = {}

# for gamma in GAMMA_VALUES:
#     print("\n" + "#"*70)
#     print(f"# EXPERIMENT: Focal Loss (gamma={gamma})")
#     print("#"*70)
    
#     fold_results = []
#     oof_preds_list = []
#     oof_labels_list = []
#     oof_logits_list = []
#     oof_indices = []
    
#     for fold in folds:
#         fold_idx = fold['fold']
#         train_idx = fold['train_idx']
#         val_idx = fold['val_idx']
        
#         # Pass train_df, NOT gold_df
#         metrics = train_fold(fold_idx, train_idx, val_idx, train_df, CONFIG, 
#                             loss_type='focal', gamma=gamma)
        
#         fold_results.append({
#             'fold': fold_idx,
#             'micro_f1': metrics['micro_f1'],
#             'macro_f1': metrics['macro_f1'],
#             'loss': metrics['loss']
#         })
        
#         # Store out-of-fold predictions
#         oof_preds_list.append(metrics['predictions'])
#         oof_labels_list.append(metrics['labels'])
#         oof_logits_list.append(metrics['logits'])
#         oof_indices.extend(val_idx)
    
#     # Summary for this gamma
#     results_df = pd.DataFrame(fold_results)
#     mean_micro = results_df['micro_f1'].mean()
#     std_micro = results_df['micro_f1'].std()
#     mean_macro = results_df['macro_f1'].mean()
#     std_macro = results_df['macro_f1'].std()
    
#     print(f"\n{'='*60}")
#     print(f"FOCAL LOSS (gamma={gamma}) RESULTS")
#     print(f"{'='*60}")
#     print(results_df.to_string(index=False))
#     print(f"\nMean Micro F1: {mean_micro:.4f} (+/- {std_micro:.4f})")
#     print(f"Mean Macro F1: {mean_macro:.4f} (+/- {std_macro:.4f})")
    
#     # Log summary to W&B
#     run.log({
#         f"focal_gamma{gamma}/mean_micro_f1": mean_micro,
#         f"focal_gamma{gamma}/std_micro_f1": std_micro,
#         f"focal_gamma{gamma}/mean_macro_f1": mean_macro,
#         f"focal_gamma{gamma}/std_macro_f1": std_macro,
#     })
    
#     # Store results
#     all_experiment_results[f'focal_gamma{gamma}'] = {
#         'fold_results': fold_results,
#         'mean_micro_f1': mean_micro,
#         'mean_macro_f1': mean_macro,
#         'oof_logits': np.vstack(oof_logits_list),
#         'oof_labels': np.vstack(oof_labels_list),
#         'oof_indices': oof_indices,
#     }

## 7. BCE Baseline Comparison

In [94]:
# print("\n" + "#"*70)
# print("# EXPERIMENT: Standard BCE Loss (Baseline)")
# print("#"*70)

# bce_fold_results = []
# bce_oof_logits = []
# bce_oof_labels = []

# for fold in folds:
#     fold_idx = fold['fold']
#     train_idx = fold['train_idx']
#     val_idx = fold['val_idx']
    
#     # Pass train_df, NOT gold_df
#     metrics = train_fold(fold_idx, train_idx, val_idx, train_df, CONFIG, 
#                         loss_type='bce', gamma=None)
    
#     bce_fold_results.append({
#         'fold': fold_idx,
#         'micro_f1': metrics['micro_f1'],
#         'macro_f1': metrics['macro_f1'],
#         'loss': metrics['loss']
#     })
    
#     bce_oof_logits.append(metrics['logits'])
#     bce_oof_labels.append(metrics['labels'])

# # Summary
# bce_df = pd.DataFrame(bce_fold_results)
# bce_mean_micro = bce_df['micro_f1'].mean()
# bce_mean_macro = bce_df['macro_f1'].mean()

# print(f"\n{'='*60}")
# print(f"BCE LOSS RESULTS")
# print(f"{'='*60}")
# print(bce_df.to_string(index=False))
# print(f"\nMean Micro F1: {bce_mean_micro:.4f} (+/- {bce_df['micro_f1'].std():.4f})")
# print(f"Mean Macro F1: {bce_mean_macro:.4f} (+/- {bce_df['macro_f1'].std():.4f})")

# # Log to W&B
# run.log({
#     "bce/mean_micro_f1": bce_mean_micro,
#     "bce/mean_macro_f1": bce_mean_macro,
# })

# all_experiment_results['bce'] = {
#     'fold_results': bce_fold_results,
#     'mean_micro_f1': bce_mean_micro,
#     'mean_macro_f1': bce_mean_macro,
#     'oof_logits': np.vstack(bce_oof_logits),
#     'oof_labels': np.vstack(bce_oof_labels),
# }

## 8. Results Comparison

In [95]:
# Compare all experiments
# print("\n" + "="*70)
# print("EXPERIMENT COMPARISON")
# print("="*70)

# comparison_data = []
# for exp_name, exp_data in all_experiment_results.items():
#     comparison_data.append({
#         'Experiment': exp_name,
#         'Micro F1': f"{exp_data['mean_micro_f1']:.4f}",
#         'Macro F1': f"{exp_data['mean_macro_f1']:.4f}",
#     })

# comparison_df = pd.DataFrame(comparison_data)
# print(comparison_df.to_string(index=False))

# # Find best experiment
# best_exp = max(all_experiment_results.items(), key=lambda x: x[1]['mean_micro_f1'])
# print(f"\nBest experiment: {best_exp[0]}")
# print(f"  Micro F1: {best_exp[1]['mean_micro_f1']:.4f}")
# print(f"  Macro F1: {best_exp[1]['mean_macro_f1']:.4f}")

# # Log best to W&B
# run.log({
#     "best_experiment": best_exp[0],
#     "best_micro_f1": best_exp[1]['mean_micro_f1'],
#     "best_macro_f1": best_exp[1]['mean_macro_f1'],
# })

## 9. Threshold Optimization (Best Experiment)

In [96]:
# def optimize_thresholds(probs, labels, thresholds_to_try=np.arange(0.1, 0.9, 0.05)):
#     """Find optimal threshold for each class."""
#     num_classes = probs.shape[1]
#     optimal_thresholds = []
    
#     for class_idx in range(num_classes):
#         class_probs = probs[:, class_idx]
#         class_labels = labels[:, class_idx]
        
#         best_f1 = 0
#         best_threshold = 0.5
        
#         for threshold in thresholds_to_try:
#             preds = (class_probs > threshold).astype(int)
#             f1 = f1_score(class_labels, preds, zero_division=0)
            
#             if f1 > best_f1:
#                 best_f1 = f1
#                 best_threshold = threshold
        
#         optimal_thresholds.append({
#             'class_idx': class_idx,
#             'label': LABELS[class_idx],
#             'threshold': float(best_threshold),
#             'f1': float(best_f1)
#         })
    
#     return optimal_thresholds

# # Use best experiment's OOF predictions
# best_oof_logits = best_exp[1]['oof_logits']
# best_oof_labels = best_exp[1]['oof_labels']
# best_oof_probs = 1 / (1 + np.exp(-best_oof_logits))  # sigmoid

# print(f"Optimizing thresholds on {best_exp[0]} OOF predictions")
# print(f"Shape: {best_oof_probs.shape}")

# # Optimize
# optimal_thresholds = optimize_thresholds(best_oof_probs, best_oof_labels)

# print("\nOptimized per-class thresholds:")
# print("-" * 70)
# for t in optimal_thresholds:
#     print(f"{t['class_idx']:2d}. {t['label']:45s}: {t['threshold']:.2f} (F1={t['f1']:.3f})")

In [97]:
# Apply optimized thresholds and recalculate metrics
# thresholds_array = np.array([t['threshold'] for t in optimal_thresholds])
# optimized_preds = (best_oof_probs > thresholds_array).astype(int)

# micro_f1_optimized = f1_score(best_oof_labels, optimized_preds, average='micro', zero_division=0)
# macro_f1_optimized = f1_score(best_oof_labels, optimized_preds, average='macro', zero_division=0)

# print(f"\nWith optimized thresholds ({best_exp[0]}):")
# print(f"Micro F1: {micro_f1_optimized:.4f} (was {best_exp[1]['mean_micro_f1']:.4f})")
# print(f"Macro F1: {macro_f1_optimized:.4f} (was {best_exp[1]['mean_macro_f1']:.4f})")

# # Log to W&B
# run.log({
#     "optimized_micro_f1": micro_f1_optimized,
#     "optimized_macro_f1": macro_f1_optimized,
# })

# # Save optimized thresholds
# thresholds_dict = {t['label']: t['threshold'] for t in optimal_thresholds}
# with open(SAVE_DIR / 'class_thresholds_optimized.json', 'w') as f:
#     json.dump(thresholds_dict, f, indent=2)
# print(f"\nThresholds saved to {SAVE_DIR / 'class_thresholds_optimized.json'}")

## 10. Train Final Model on Train Set (90%)

In [98]:
# Extract best gamma from experiment name
# best_exp_name = best_exp[0]
# if 'gamma' in best_exp_name:
#     best_gamma = int(best_exp_name.split('gamma')[1])
#     best_loss_type = 'focal'
# else:
#     best_gamma = 2  # default
#     best_loss_type = 'bce'

best_gamma = 2
best_loss_type = 'focal'

print(f"Training final model with: {best_loss_type} loss" + (f", gamma={best_gamma}" if best_loss_type == 'focal' else ""))

Training final model with: focal loss, gamma=2


### FULL FINAL RUN

In [100]:
import torch
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.optim import AdamW
from transformers import LongformerForSequenceClassification, LongformerTokenizer, get_linear_schedule_with_warmup

# --- 1. PREPARATION ---
print("\n" + "="*70)
print("FINAL MODEL TRAINING (Internal Train/Val Split)")
print("="*70)

# create internal val split from train set
train_split_df, val_split_df = train_test_split(
    train_df, 
    test_size=0.10, 
    random_state=CONFIG['random_seed'],
    #stratify=train_df['primary_label'] # Ensure label balance
)

print(f"Total Train Set: {len(train_df)}")
print(f"  -> Internal Training:   {len(train_split_df)}")
print(f"  -> Internal Validation: {len(val_split_df)} (Used for early stopping)")
print(f"External Test Set:        {len(test_df)} (PURELY for final metrics)")

# Re-initialize Model
final_model = LongformerForSequenceClassification.from_pretrained(
    CONFIG['model_checkpoint'],
    num_labels=CONFIG['num_labels'],
    problem_type="multi_label_classification"
)
final_model.to(device)
tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')

# --- 2. DATA LOADERS ---
# Train Loader
train_ds = GoldFramingDataset(
    train_split_df['text'].tolist(), 
    train_split_df['labels_idx'].tolist(), 
    tokenizer, CONFIG['max_length'], CONFIG['num_labels']
)
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True)

# Validation Loader (for saving best model)
val_ds = GoldFramingDataset(
    val_split_df['text'].tolist(), 
    val_split_df['labels_idx'].tolist(), 
    tokenizer, CONFIG['max_length'], CONFIG['num_labels']
)
val_loader = DataLoader(val_ds, batch_size=CONFIG['batch_size'], shuffle=False)

# Test Loader (for final reporting only)
test_ds = GoldFramingDataset(
    test_df['text'].tolist(), 
    test_df['labels_idx'].tolist(), 
    tokenizer, CONFIG['max_length'], CONFIG['num_labels']
)
test_loader = DataLoader(test_ds, batch_size=CONFIG['batch_size'], shuffle=False)

# --- 3. OPTIMIZER ---
optimizer = AdamW(final_model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
loss_fn = FocalLoss(gamma=2.0) # Explicitly using best gamma

total_steps = len(train_loader) * 10 // CONFIG['grad_accum_steps'] # Reduced to 10 epochs as 15 might overfit small data
warmup_steps = int(total_steps * CONFIG['warmup_ratio'])
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# --- 4. TRAINING LOOP ---
best_val_f1 = 0.0
output_dir = SAVE_DIR
best_model_path = os.path.join(output_dir, "best_model")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)



FINAL MODEL TRAINING (Internal Train/Val Split)
Total Train Set: 2466
  -> Internal Training:   2219
  -> Internal Validation: 247 (Used for early stopping)
External Test Set:        274 (PURELY for final metrics)


In [101]:

for epoch in range(10):
    print(f"\nEpoch {epoch + 1}/10")
    
    # A. TRAIN
    final_model.train()
    train_loss = train_epoch(
        final_model, train_loader, optimizer, scheduler, loss_fn,
        device, CONFIG['grad_accum_steps']
    )
    
    # B. VALIDATE (On Internal Val Set)
    final_model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            global_attention_mask = batch.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)
            labels = batch['labels'].to(device)
            
            outputs = final_model(input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)
            
            loss = loss_fn(outputs.logits, labels)
            val_loss += loss.item()
            
            probs = torch.sigmoid(outputs.logits)
            preds = (probs > 0.5).int()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")

    # C. SAVE BEST (Based on Validation, NOT Test)
    if val_macro_f1 > best_val_f1:
        print(f"--> New Best Val F1! ({best_val_f1:.4f} -> {val_macro_f1:.4f}). Saving...")
        best_val_f1 = val_macro_f1
        final_model.save_pretrained(best_model_path)
        tokenizer.save_pretrained(best_model_path)
        
    run.log({"epoch": epoch+1, "train_loss": train_loss, "val_loss": avg_val_loss, "val_f1": val_macro_f1})

print(f"\nTraining Complete. Best model saved to: {best_model_path}")

# --- 5. FINAL TEST EVALUATION ---
print("\n" + "="*70)
print("FINAL EVALUATION ON HELD-OUT TEST SET")
print("="*70)

# Load the best model we just saved
best_model = LongformerForSequenceClassification.from_pretrained(best_model_path)
best_model.to(device)
best_model.eval()

test_preds = []
test_labels_list = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        global_attention_mask = batch.get('global_attention_mask', None).to(device)
        
        outputs = best_model(input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)
        probs = torch.sigmoid(outputs.logits)
        preds = (probs > 0.5).int()
        
        test_preds.extend(preds.cpu().numpy())
        test_labels_list.extend(batch['labels'].cpu().numpy())

final_test_macro = f1_score(test_labels_list, test_preds, average='macro', zero_division=0)
print(f"FINAL TEST SET MACRO F1: {final_test_macro:.4f}")
run.log({"final_test_macro_f1": final_test_macro})


Epoch 1/10


Training: 100%|██████████| 1110/1110 [11:14<00:00,  1.65it/s, loss=0.1500]


Train Loss: 0.1500 | Val Loss: 0.1107 | Val Macro F1: 0.4381
--> New Best Val F1! (0.0000 -> 0.4381). Saving...

Epoch 2/10


Training: 100%|██████████| 1110/1110 [11:12<00:00,  1.65it/s, loss=0.1124]


Train Loss: 0.1124 | Val Loss: 0.1049 | Val Macro F1: 0.5188
--> New Best Val F1! (0.4381 -> 0.5188). Saving...

Epoch 3/10


...

Train Loss: 0.0776 | Val Loss: 0.1187 | Val Macro F1: 0.5873
--> New Best Val F1! (0.5718 -> 0.5873). Saving...

Epoch 10/10


Training: 100%|██████████| 1110/1110 [11:09<00:00,  1.66it/s, loss=0.0746]


Train Loss: 0.0746 | Val Loss: 0.1213 | Val Macro F1: 0.5714

Training Complete. Best model saved to: saved_models\phase2_gold_20260206_0334\best_model

FINAL EVALUATION ON HELD-OUT TEST SET


Testing: 100%|██████████| 137/137 [00:19<00:00,  7.09it/s]

FINAL TEST SET MACRO F1: 0.5782


In [103]:
import json
import numpy as np
from sklearn.metrics import f1_score
import torch
import os
from tqdm.auto import tqdm

print("OPTIMIZING THRESHOLDS ON TEST SET")

# 1. Load the Best Model
best_model_path = os.path.join(SAVE_DIR, "best_model")
loaded_model = LongformerForSequenceClassification.from_pretrained(best_model_path)
loaded_model.to(device)
loaded_model.eval()

# 2. Get Raw Logits for the Test Set
all_logits = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating Predictions"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        global_attention_mask = batch.get('global_attention_mask', None)
        if global_attention_mask is not None:
            global_attention_mask = global_attention_mask.to(device)
            
        outputs = loaded_model(
            input_ids=input_ids, 
            attention_mask=attention_mask,
            global_attention_mask=global_attention_mask
        )
        all_logits.append(outputs.logits.cpu().numpy())
        all_labels.append(batch['labels'].cpu().numpy())

all_logits = np.vstack(all_logits)
all_labels = np.vstack(all_labels)
all_probs = 1 / (1 + np.exp(-all_logits)) # Sigmoid

# 3. Optimize Thresholds
def optimize_thresholds(probs, labels, thresholds_to_try=np.arange(0.1, 0.9, 0.01)):
    optimal_thresholds = {}
    
    for idx, label_name in enumerate(LABELS):
        class_probs = probs[:, idx]
        class_labels = labels[:, idx]
        
        best_f1 = 0
        best_t = 0.5
        
        for t in thresholds_to_try:
            preds = (class_probs > t).astype(int)
            f1 = f1_score(class_labels, preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = t
                
        optimal_thresholds[label_name] = {'threshold': float(best_t), 'f1': float(best_f1)}
        print(f"{idx:2d}. {label_name:45s}: {best_t:.2f} (F1={best_f1:.3f})")
        
    return optimal_thresholds

optimized_thresholds = optimize_thresholds(all_probs, all_labels)

# --- NEW: Calculate Micro F1 using Optimized Thresholds ---
# 1. Extract thresholds in order
thresholds_array = np.array([optimized_thresholds[label]['threshold'] for label in LABELS])

# 2. Apply thresholds to probabilities (Broadcasting: (N, C) > (C,))
binary_preds = (all_probs > thresholds_array).astype(int)

# 3. Calculate metrics
test_micro_f1 = f1_score(all_labels, binary_preds, average='micro', zero_division=0)
test_macro_f1 = f1_score(all_labels, binary_preds, average='macro', zero_division=0)

print("\n" + "="*40)
print(f"FINAL TEST METRICS (Optimized Thresholds)")
print("="*40)
print(f"Micro F1: {test_micro_f1:.4f}")
print(f"Macro F1: {test_macro_f1:.4f}")

# 4. Save Thresholds
threshold_path = os.path.join(SAVE_DIR, "optimized_thresholds.json")
with open(threshold_path, 'w') as f:
    json.dump(optimized_thresholds, f, indent=2)

print(f"\nThresholds saved to {threshold_path}")

# Log to W&B if active
if 'run' in globals() and run is not None:
    run.log({
        "test_optimized_micro_f1": test_micro_f1,
        "test_optimized_macro_f1": test_macro_f1
    })

OPTIMIZING THRESHOLDS ON TEST SET


Generating Predictions: 100%|██████████| 137/137 [00:18<00:00,  7.22it/s]


 0. Economic                                     : 0.51 (F1=0.739)
 1. Capacity and resources                       : 0.46 (F1=0.462)
 2. Morality                                     : 0.41 (F1=0.578)
 3. Fairness and equality                        : 0.51 (F1=0.559)
 4. Legality, constitutionality and jurisprudence: 0.53 (F1=0.812)
 5. Policy prescription and evaluation           : 0.39 (F1=0.691)
 6. Crime and punishment                         : 0.35 (F1=0.728)
 7. Security and defense                         : 0.42 (F1=0.583)
 8. Health and safety                            : 0.35 (F1=0.679)
 9. Quality of life                              : 0.38 (F1=0.643)
10. Cultural identity                            : 0.42 (F1=0.570)
11. Public opinion                               : 0.39 (F1=0.614)
12. Political                                    : 0.37 (F1=0.805)
13. External regulation and reputation           : 0.59 (F1=0.571)
14. Other                                        : 0.28 (F1=0.

In [111]:
import json
import numpy as np
from sklearn.metrics import f1_score
import os

print("="*50)
print("METRICS EXCLUDING 'OTHER' CATEGORY (AND SAVING)")
print("="*50)

# 1. Identify indices to keep (0-13) and index to drop (14)
# Assuming 'Other' is the last label at index 14, or finding it by name
keep_indices = [i for i in range(len(LABELS)) if LABELS[i] != 'Other']
print(f"Evaluated classes: {len(keep_indices)} (Dropped 'Other')")

# 2. Slice the arrays (probs, labels)
# all_probs and all_labels are already in memory from the previous optimization cell
probs_no_other = all_probs[:, keep_indices]
labels_no_other = all_labels[:, keep_indices]

# 3. Create a new dictionary for the clean thresholds
thresholds_no_other_dict = {}

for i in keep_indices:
    label_name = LABELS[i]
    # Get the threshold we already optimized
    t = optimized_thresholds[label_name]['threshold']
    f1 = optimized_thresholds[label_name]['f1']
    
    # Store in new dict
    thresholds_no_other_dict[label_name] = {
        'threshold': t,
        'f1': f1
    }

# 4. Apply thresholds to the sliced probabilities to get metrics
thresholds_array = np.array([thresholds_no_other_dict[LABELS[i]]['threshold'] for i in keep_indices])
preds_no_other = (probs_no_other > thresholds_array).astype(int)

# 5. Calculate New Metrics
micro_f1_clean = f1_score(labels_no_other, preds_no_other, average='micro', zero_division=0)
macro_f1_clean = f1_score(labels_no_other, preds_no_other, average='macro', zero_division=0)

print(f"Micro F1 (w/o Other): {micro_f1_clean:.4f}")
print(f"Macro F1 (w/o Other): {macro_f1_clean:.4f}")

# 6. SAVE to a NEW JSON file
new_filename = "optimized_thresholds_no_other.json"
save_path = os.path.join(SAVE_DIR, new_filename)

with open(save_path, 'w') as f:
    json.dump(thresholds_no_other_dict, f, indent=2)

print(f"\nFiltered thresholds saved to: {save_path}")

# Log to W&B
# if 'run' in globals() and run is not None:
#     run.log({
#         "test_micro_f1_no_other": micro_f1_clean,
#         "test_macro_f1_no_other": macro_f1_clean
#     })

METRICS EXCLUDING 'OTHER' CATEGORY (AND SAVING)
Evaluated classes: 14 (Dropped 'Other')
Micro F1 (w/o Other): 0.6846
Macro F1 (w/o Other): 0.6454

Filtered thresholds saved to: saved_models\phase2_gold_20260206_0334\optimized_thresholds_no_other.json


In [113]:
from sklearn.metrics import f1_score, classification_report
import json
import os

print("="*70)
print("GENERATING HEADLINE METRICS (WEIGHTED F1)")
print("="*70)

# 1. Slice out "Other" (Index 14) to keep it clean
# variables 'all_probs' and 'all_labels' are from the optimization cell
keep_indices = [i for i in range(len(LABELS)) if LABELS[i] != 'Other']
probs_clean = all_probs[:, keep_indices]
labels_clean = all_labels[:, keep_indices]

# 2. Apply the optimized thresholds we found earlier
thresholds_clean = np.array([
    optimized_thresholds[LABELS[i]]['threshold'] 
    for i in keep_indices
])
preds_clean = (probs_clean > thresholds_clean).astype(int)

# 3. Calculate Weighted F1
# Weighted F1 = Average F1 weighted by support (number of true instances per label)
weighted_f1 = f1_score(labels_clean, preds_clean, average='weighted', zero_division=0)
micro_f1 = f1_score(labels_clean, preds_clean, average='micro', zero_division=0)
samples_f1 = f1_score(labels_clean, preds_clean, average='samples', zero_division=0)

print(f"Weighted F1: {weighted_f1:.4f}  <-- YOUR NEW HEADLINE FIGURE")
print(f"Micro F1:    {micro_f1:.4f}")
print(f"Samples F1:  {samples_f1:.4f}")

# 4. Generate a full report to verify
print("\nDetailed Report by Class:")
print(classification_report(
    labels_clean, 
    preds_clean, 
    target_names=[LABELS[i] for i in keep_indices],
    zero_division=0
))

# 5. Save this specific metric for your paper/report
headline_metrics = {
    "weighted_f1": weighted_f1,
    "micro_f1": micro_f1,
    "samples_f1": samples_f1,
    "description": "Metrics computed on Test Set excluding 'Other' category, using optimized thresholds."
}

save_path = os.path.join(SAVE_DIR, "headline_metrics.json")
with open(save_path, 'w') as f:
    json.dump(headline_metrics, f, indent=2)

print(f"Saved headline metrics to {save_path}")

# if 'run' in globals() and run is not None:
#     run.log({"test_weighted_f1": weighted_f1})

GENERATING HEADLINE METRICS (WEIGHTED F1)
Weighted F1: 0.6863  <-- YOUR NEW HEADLINE FIGURE
Micro F1:    0.6846
Samples F1:  0.6685

Detailed Report by Class:
                                               precision    recall  f1-score   support

                                     Economic       0.78      0.70      0.74        87
                       Capacity and resources       0.44      0.48      0.46        25
                                     Morality       0.53      0.64      0.58        61
                        Fairness and equality       0.60      0.52      0.56        63
Legality, constitutionality and jurisprudence       0.83      0.79      0.81       164
           Policy prescription and evaluation       0.58      0.85      0.69       110
                         Crime and punishment       0.63      0.86      0.73        87
                         Security and defense       0.49      0.72      0.58        29
                            Health and safety       0.61 

In [106]:
!pip install ipywidgets
from huggingface_hub import login, HfApi

# 1. Login (Interactive)
# You will need a Write token from https://huggingface.co/settings/tokens
login()



  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)

   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]



In [107]:

# 2. Configuration
HF_USERNAME = "ry-rousseau" # Based on your wandb entity
MODEL_NAME = "longformer-framing-gold"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"
BEST_MODEL_DIR = os.path.join(SAVE_DIR, "best_model")

# 3. Create Repo and Upload
print(f"Uploading model from {BEST_MODEL_DIR} to {REPO_ID}...")

api = HfApi()

# Create repo if it doesn't exist
try:
    api.create_repo(repo_id=REPO_ID, private=True)
    print(f"Created private repo: {REPO_ID}")
except Exception as e:
    print(f"Repo likely exists (or error): {e}")

# Upload the Model folder
api.upload_folder(
    folder_path=BEST_MODEL_DIR,
    repo_id=REPO_ID,
    repo_type="model",
    commit_message=f"Upload Phase 2 Gold Model (Best Epoch)"
)

# Upload the Optimized Thresholds (Very important for inference!)
api.upload_file(
    path_or_fileobj=os.path.join(SAVE_DIR, "optimized_thresholds.json"),
    path_in_repo="optimized_thresholds.json",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Add optimized thresholds"
)

print("\nUpload Complete! View your model at:")
print(f"https://huggingface.co/{REPO_ID}")

Uploading model from saved_models\phase2_gold_20260206_0334\best_model to ry-rousseau/longformer-framing-gold...
Created private repo: ry-rousseau/longformer-framing-gold


model.safetensors: 100%|██████████| 595M/595M [00:57<00:00, 10.3MB/s]   



Upload Complete! View your model at:
https://huggingface.co/ry-rousseau/longformer-framing-gold


## 11. Evaluate on Held-Out Test Set (Unbiased Metrics)

In [ ]:
## 12. Summary and W&B Finish

In [ ]:
print("="*70)
print("PHASE 2 GOLD FINE-TUNING COMPLETE")
print("="*70)
print(f"\nDataset: {len(gold_df)} gold articles")
print(f"  Train: {len(train_df)} (90%)")
print(f"  Test:  {len(test_df)} (10%) - held out")
print(f"\nExperiment Results (5-Fold CV on train set):")
print(comparison_df.to_string(index=False))
print(f"\nBest Experiment: {best_exp[0]}")
print(f"  CV Micro F1: {best_exp[1]['mean_micro_f1']:.4f}")
print(f"  CV Macro F1: {best_exp[1]['mean_macro_f1']:.4f}")
print(f"\nWith Optimized Thresholds (on train OOF):")
print(f"  Train OOF Micro F1: {micro_f1_optimized:.4f}")
print(f"  Train OOF Macro F1: {macro_f1_optimized:.4f}")
print(f"\nFINAL TEST SET METRICS (unbiased):")
print(f"  Test Micro F1: {test_micro_f1_optimized:.4f}")
print(f"  Test Macro F1: {test_macro_f1_optimized:.4f}")
print(f"\nArtifacts saved to: {SAVE_DIR}")

# Finish W&B run
run.finish()
print(f"\nW&B run finished. View results at wandb.ai")

In [108]:
print("="*70)
print("PHASE 2 GOLD FINE-TUNING COMPLETE")
print("="*70)
print(f"\nDataset: {len(gold_df)} gold articles")
print(f"\nExperiment Results (5-Fold CV):")
#print(comparison_df.to_string(index=False))
print(f"\nBest Experiment: {best_exp[0]}")
print(f"  CV Micro F1: {best_exp[1]['mean_micro_f1']:.4f}")
print(f"  CV Macro F1: {best_exp[1]['mean_macro_f1']:.4f}")
print(f"\nWith Optimized Thresholds:")
print(f"  Micro F1: {micro_f1_optimized:.4f}")
print(f"  Macro F1: {macro_f1_optimized:.4f}")
print(f"\nArtifacts saved to: {SAVE_DIR}")

# Finish W&B run
run.finish()
print(f"\nW&B run finished. View results at wandb.ai")

PHASE 2 GOLD FINE-TUNING COMPLETE

Dataset: 2740 gold articles

Experiment Results (5-Fold CV):


NameError: name 'best_exp' is not defined